# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [4]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [5]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os
import chromadb
import json
from chromadb.utils import embedding_functions
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from chromadb.errors import NotFoundError
#from lib.vector_db import VectorStoreManager, CorpusLoaderService
#from lib.rag import RAG

In [6]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
load_dotenv('/workspace/Code/.env')
assert os.getenv('OPENAI_API_KEY') is not None
assert os.getenv('TAVILY_API_KEY') is not None

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [7]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

CHROMA_PATH = ".chroma-db"
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(api_key=os.getenv("OPENAI_API_KEY"))
COLLECTION_NAME = "all-games-data"
chroma_client = chromadb.PersistentClient(CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME, embedding_function=embedding_fn)

from typing import List, Dict, Any

def make_retrieve_game_tool(collection):
    @tool
    def retrieve_game(query: str, n_results: int = 5) -> List[Dict[str, Any]]:
        print(f"DEBUG n_results = {n_results}")

        res = collection.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )

        ids = res.get("ids", [[]])[0] or []
        docs = res.get("documents", [[]])[0] or []
        metas = res.get("metadatas", [[]])[0] or []
        dists = res.get("distances", [[]])[0] or []

        out: List[Dict[str, Any]] = []
        for i in range(len(ids)):
            meta = metas[i] or {}
            out.append({
                "id": ids[i],
                "Name": meta.get("Name"),
                "Platform": meta.get("Platform"),
                "YearOfRelease": meta.get("YearOfRelease"),
                "Description": meta.get("Description"),
                "distance": dists[i] if i < len(dists) else None,
                "document": docs[i] if i < len(docs) else None,
            })

        print("DEBUG retrieve_game returned first:", out[:1])
        return json.dumps(out)

    return retrieve_game


Failed to send telemetry event ClientStartEvent: module 'chromadb' has no attribute 'get_settings'


#### Evaluate Retrieval Tool

In [8]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

import json
from typing import Any, Dict, List, Optional
from pydantic import BaseModel, Field
from openai import OpenAI

client = OpenAI()

class EvaluationReport(BaseModel):
    useful: bool = Field(..., description="Whether the retrieved docs are sufficient to answer the question.")
    description: str = Field(..., description="Actionable explanation: what is missing / why sufficient / how to improve retrieval.")

def _format_retrieved_docs(retrieved_docs: List[Dict[str, Any]], max_docs: int = 8, max_chars: int = 900) -> str:
    """
    Turn your retrieve_game output (list of dicts) into compact text for the judge.
    """
    lines = []
    for i, d in enumerate(retrieved_docs[:max_docs], start=1):
        platform = str(d.get("Platform", "")).strip()
        name = str(d.get("Name", "")).strip()
        year = str(d.get("YearOfRelease", "")).strip()
        desc = str(d.get("Description", "")).strip()

        block = f"Doc {i}: [{platform}] {name} ({year}) - {desc}".strip()
        if len(block) > max_chars:
            block = block[:max_chars] + "…"
        lines.append(block)

    return "\n".join(lines).strip()

@tool
def evaluate_retrieval(question: str, retrieved_docs: Any) -> Dict[str, Any]:
    """
    Evaluates whether retrieved documents are sufficient to answer the question.
    Accepts YEAR-level answers for questions asking 'which year' / 'when'.
    """

    # Debug: confirm what we actually received
    try:
        ln = len(retrieved_docs) if retrieved_docs is not None else 0
    except Exception:
        ln = -1
    print("DEBUG evaluate_retrieval type:", type(retrieved_docs), "len:", ln)

    # If retriever accidentally returned a JSON string, try parsing it
    if isinstance(retrieved_docs, str):
        try:
            retrieved_docs = json.loads(retrieved_docs)
            print("DEBUG evaluate_retrieval parsed JSON string")
        except Exception:
            return {
                "useful": True,
                "description": "Docs arrived as plain text (agent stringified tool output), but still usable."
            }

    # Must be a non-empty list of dicts
    if not retrieved_docs or not isinstance(retrieved_docs, list):
        return {
            "useful": False,
            "description": "No structured documents were provided to the evaluator."
        }

    q = question.lower()
    asks_year = (
        "which year" in q
        or ("year" in q and "release" in q)
        or ("released" in q and "year" in q)
        or q.startswith("when")
    )

    # Year-based questions → YearOfRelease is sufficient
    if asks_year:
        for doc in retrieved_docs:
            if isinstance(doc, dict):
                year = doc.get("YearOfRelease")
                if year not in (None, "", "Unknown"):
                    return {
                        "useful": True,
                        "description": "YearOfRelease found in retrieved metadata."
                    }

        return {
            "useful": False,
            "description": "YearOfRelease not present in retrieved metadata."
        }

    # Default: allow answer if we retrieved anything at all
    return {
        "useful": True,
        "description": "Retrieved documents appear sufficient."
    }


#### Game Web Search Tool

In [9]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 


from tavily import TavilyClient
from typing import Any, Dict

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def game_web_search(question: str, max_results: int = 5) -> str:
    """
    Web search for game-industry questions when vector DB results are insufficient.
    args:
      - question: user question
      - max_results: number of results (default 5)
    returns:
      - JSON string: list of {title, url, content, score}
    """
    res = tavily.search(
        query=question,
        max_results=max_results,
        search_depth="advanced",   # good default
        include_answer=False
    )

    results = []
    for r in res.get("results", []):
        results.append({
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content"),
            "score": r.get("score"),
        })

    return json.dumps(results)

### Agent

In [33]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

retrieve_game = make_retrieve_game_tool(collection)

agent = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game,evaluate_retrieval,game_web_search],
    instructions=("""
    You are a helpful assistant for questions about video games.

    Follow this process exactly:
    0. First decide if the question is about video games or the video game industry.
    - If NOT, reply exactly: "I can only answer video game and video game industry questions."
    - Do NOT call any tools and do NOT answer anything else.
    1. Call retrieve_game with the user question.
    2. Store the result as `retrieved_docs`.
    3. Call evaluate_retrieval using:
    - question = the original user question
    - retrieved_docs = the output from retrieve_game
    4. If evaluation.useful is true:
    - Answer the question using ONLY retrieved_docs.
    5. If evaluation.useful is false:
    - Search information related to games only on the web by only using game_web_search
    - when question is not realted to games, do not answer
    - show information when returned
    - State the source of information.

    Rules:
    - Never call evaluate_retrieval without retrieved_docs.
    - Never invent games.
    - If retrieve_game returns an empty list, ask one clarifying question.
    - If the question is not about video games, you MUST refuse with the exact sentence above.
    """.strip())
)

In [34]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
import time
runner = Run(agent,start_timestamp=time.time())
query = [
    "When was titanic movie released?"
]

for q in query:
    print(f"\nQ: {q}")
    run=agent.invoke(q)
    final_state=run.get_final_state()
    print(f"\nQ: {q}")
    print("A:", final_state["messages"][-1].content)


Q: When was titanic movie released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Q: When was titanic movie released?
A: I can only answer video game and video game industry questions.


### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes